### Exercises

#### Exercise 1

Check to see if any of the players are duplicated (appear more than one time in the dataset).

In [158]:
import pandas as pd

df = pd.read_csv("../data/fifa.csv", low_memory=False)

longname_duplicate = df[df["LongName"].duplicated()]["LongName"]
df[df["LongName"] == longname_duplicate.iloc[0]]

,photoUrl,LongName,playerUrl,Nationality,Positions,Name,Age,↓OVA,POT,Team & Contract,...,A/W,D/W,IR,PAC,SHO,PAS,DRI,DEF,PHY,Hits
899,https://cdn.sofifa.com/players/251/698/21_60.png,Kevin Berlaso,http://sofifa.com/player/251698/kevin-berlaso/...,Ecuador,RB,K. Berlaso,32,77,77,\n Ecuador\nFree\n\n,...,High,Medium,2 ★,78,56,69,77,72,68,12
944,https://cdn.sofifa.com/players/251/698/21_60.png,Kevin Berlaso,http://sofifa.com/player/251698/kevin-berlaso/...,Ecuador,RB,K. Berlaso,32,77,77,\n Ecuador\nFree\n\n,...,High,Medium,2 ★,78,56,69,77,72,68,12


#### Exercise 2

Inspect the dataset to see if there are any problems with missing data. If there are, determine how you would like to handle these missing values and then proceed to implement your chosen method.

Check for missing data

In [159]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18979 entries, 0 to 18978
Data columns (total 77 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   photoUrl          18979 non-null  object
 1   LongName          18979 non-null  object
 2   playerUrl         18979 non-null  object
 3   Nationality       18979 non-null  object
 4   Positions         18979 non-null  object
 5   Name              18979 non-null  object
 6   Age               18979 non-null  int64 
 7   ↓OVA              18979 non-null  int64 
 8   POT               18979 non-null  int64 
 9   Team & Contract   18979 non-null  object
 10  ID                18979 non-null  int64 
 11  Height            18979 non-null  object
 12  Weight            18979 non-null  object
 13  foot              18979 non-null  object
 14  BOV               18979 non-null  int64 
 15  BP                18979 non-null  object
 16  Growth            18979 non-null  int64 
 17  Joined      

#### Exercise 3

Use mean imputation to replace missing values in the Hits column.

In [160]:
k_to_1000 = 1000

multiplier_hits = df["Hits"].str.contains("K").map(
    lambda x: k_to_1000 if x else 1
)

df["Hits"] = df["Hits"].str.replace("K", "", regex=False).astype("float")*multiplier_hits
df["Hits"] = df["Hits"].fillna(df["Hits"].mean())

In [162]:
df["Hits"].isnull().sum()

0

#### Exercise 4

Remove unwanted text from the Weight column.

In [163]:
df["Weight"] = df["Weight"].str.replace("lbs", "")
df["Weight"].describe()

count     18979
unique       56
top         154
freq       1496
Name: Weight, dtype: object

#### Exercise 5

Remove unwanted text from the Value and Wage columns.

Remove unwanted text from Value first

In [164]:
df["Value"] = df["Value"].str.replace("€", "")

df["Value"] = df["Value"].str.replace("K", "")
df["Value"] = df["Value"].str.replace("M", "")

df["Value"].describe()

count     18979
unique      201
top         1.1
freq        467
Name: Value, dtype: object

Now remove unwanted text from Wage column

In [165]:
df["Wage"] = df["Wage"].str.replace("€", "")
df["Wage"] = df["Wage"].str.replace("K", "")

df["Wage"].describe()

count     18979
unique      141
top           2
freq       2997
Name: Wage, dtype: object

#### Exercise 6

Convert the Weight, Value, and Wage columns to the `float` data type.

In [167]:
df[["Weight", "Value", "Wage"]] = df[["Weight", "Value", "Wage"]].astype("float")
df[["Weight", "Value", "Wage"]].describe()

,Weight,Value,Wage
count,18979.00000,18979.000000,18979.000000
mean,165.38785,268.641393,131.795932
std,15.58793,286.116397,246.867833
min,110.00000,0.000000,0.000000
25%,154.00000,4.850000,2.000000
50%,165.00000,170.000000,6.000000
75%,176.00000,475.000000,41.000000
max,243.00000,975.000000,950.000000


#### Exercise 7

Utilize method chaining to do the following in a single pass:

1. Remove unwanted characters from the Release Clause column.
2. Convert data type to `float`
3. Impute missing values using median

In [153]:
df["Release Clause"] = df["Release Clause"].str.replace("€", "").str.replace("M", "").str.replace("K", "", regex=False).astype("float").pipe(lambda x:x.fillna(x.median()))

df["Release Clause"].describe()

count    18979.000000
mean       217.260683
std        297.001381
min          0.000000
25%          1.900000
50%         13.600000
75%        406.000000
max        999.000000
Name: Release Clause, dtype: float64

#### Exercise 8

In the Height column, determine which observations are in centimetres and which are in inches. Save the results in another variable. Then use method chaining to do the following in a single pass:

1. Remove unwanted characters from the Height column.
2. Convert data type to `float`
3. Use the previously computed variable with the inches/centimetres indicator to convert all values to inches.

In [154]:
CM_TO_INCH = 0.393701

height_multiplier = df["Height"].str.contains("centimetres").map(
    lambda x: CM_TO_INCH if x else 1
)

df["Height"] = df["Height"].str.replace("centimetres", "", ).str.replace("inches", "").astype("float")*height_multiplier

df["Height"].describe()



count    18979.000000
mean        71.348790
std          2.684461
min         60.984285
25%         69.015785
50%         71.000000
75%         73.000000
max         81.000000
Name: Height, dtype: float64

#### Exercise 9

Fix the foot column so that all values are either Left or Right.

In [155]:
foot_dict = {}

foot_dict["L"] = "Left"
foot_dict["R"] = "Right"

df["foot"] = df["foot"].map(foot_dict)
df["foot"].describe()

#### Exercise 10

Check the Wage column for any outliers and then output those observations marked as outliers.

In [168]:
df["Wage"].dtype

dtype('float64')

In [169]:
Q1 = df["Wage"].quantile(0.25)
Q3 = df["Wage"].quantile(0.75)

IQR = Q3 - Q1

df[(df["Wage"] < (Q1-1.5*IQR)) | (df["Wage"] > (Q3+1.5*IQR))].head()

,photoUrl,LongName,playerUrl,Nationality,Positions,Name,Age,↓OVA,POT,Team & Contract,...,A/W,D/W,IR,PAC,SHO,PAS,DRI,DEF,PHY,Hits
0,https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/2...,Argentina,RW ST CF,L. Messi,33,93,93,\n\n\n\nFC Barcelona\n2004 ~ 2021\n\n,...,Medium,Low,5 ★,85,92,91,95,38,65,372.0
1,https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-s...,Portugal,ST LW,Cristiano Ronaldo,35,92,92,\n\n\n\nJuventus\n2018 ~ 2022\n\n,...,High,Low,5 ★,89,93,81,89,35,77,344.0
2,https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,\n\n\n\nAtlético Madrid\n2014 ~ 2023\n\n,...,Medium,Medium,3 ★,87,92,78,90,52,90,86.0
3,https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyn...,Belgium,CAM CM,K. De Bruyne,29,91,91,\n\n\n\nManchester City\n2015 ~ 2023\n\n,...,High,High,4 ★,76,86,93,88,64,78,163.0
4,https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silv...,Brazil,LW CAM,Neymar Jr,28,91,91,\n\n\n\nParis Saint-Germain\n2017 ~ 2022\n\n,...,High,Medium,5 ★,91,85,86,94,36,59,273.0
